# sre-gym — SFT warm-start on Claude-teacher seed

Purpose: run a real SFT pass on the 39-sample Claude-teacher seed dataset so the resulting LoRA adapter can be used as the starting point for GRPO against the pool server.

What a successful run looks like:
1. All deps install without version conflicts
2. `Qwen3.5-4B-Instruct` (or `Qwen3-4B-Instruct` fallback) loads in 4-bit via Unsloth
3. `train/data/claude_seed.jsonl` loads as 39 training samples
4. 500 steps of LoRA SFT run without OOM on A100 40GB
5. `wandb` logs show loss decreasing from ~2.0 to < 0.5
6. Inference cell emits format-valid JSON for a held-out scenario prompt
7. LoRA adapter saved to `/content/sanity_ckpt/` (push to `dakshdoesdev/sre-gym-qwen35-4b-sft` as the GRPO starting point)

Upload `train/data/claude_seed.jsonl` to `/content/claude_seed.jsonl` alongside this notebook before running.

## 0. Colab runtime sanity

In [ ]:
!nvidia-smi
!python -c 'import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available())'

## 1. Install deps

In [ ]:
%%bash
# Unsloth's Colab install idiom (handles torch/xformers version pinning):
pip install -q --upgrade pip
pip install -q "unsloth[colab-new]>=2025.12,<2026.06"
pip install -q "unsloth_zoo>=2025.12,<2026.06"
pip install -q "trl>=0.12,<0.16" "transformers>=4.48,<4.60" "peft>=0.14,<0.20" "accelerate>=1.2,<2.0"
pip install -q "datasets>=3.0,<4.0" "wandb>=0.18,<1.0" "bitsandbytes>=0.45" httpx

## 2. Config

If Qwen3.5 4B fails to load, swap `MODEL_NAME` to the Qwen3 4B fallback — no other change needed.

In [ ]:
import os

# Primary target (user-selected).
MODEL_NAME = "unsloth/Qwen3.5-4B-Instruct-bnb-4bit"
# Fallback if Unsloth can't load Qwen3.5 on Colab tonight.
FALLBACK_MODEL_NAME = "unsloth/Qwen3-4B-Instruct-bnb-4bit"

MAX_SEQ_LENGTH = 4096
LORA_R = 32
LORA_ALPHA = 32
LEARNING_RATE = 2e-4
NUM_STEPS = 500        # ~78 usable samples * ~12 epochs at BATCH_SIZE=2, GRAD_ACCUM=4
BATCH_SIZE = 2
GRAD_ACCUM = 4
OUT_DIR = "/content/sanity_ckpt"
# Combined seed: Claude Opus (expert demos) + Llama-3.3-70B (realistic-agent).
# See train/data/README.md for provenance. Upload this file to /content/ before running.
SEED_JSONL_PATH = "/content/seed_combined.jsonl"

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "sre-gym-sft")
WANDB_RUN_NAME = os.environ.get("WANDB_RUN_NAME", "qwen35-4b-sft-combined-500")

os.environ.setdefault("WANDB_MODE", "online")  # flip to "offline" if no wandb login
print(f"Primary model: {MODEL_NAME}")
print(f"Seed dataset:  {SEED_JSONL_PATH}")
print(f"Training for {NUM_STEPS} steps")

## 3. Load model via Unsloth (with fallback)

In [ ]:
from unsloth import FastLanguageModel
import torch

model = None
tokenizer = None
errors = []

for candidate in (MODEL_NAME, FALLBACK_MODEL_NAME):
    try:
        print(f"Attempting to load: {candidate}")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=candidate,
            max_seq_length=MAX_SEQ_LENGTH,
            dtype=None,  # let Unsloth pick
            load_in_4bit=True,
        )
        MODEL_NAME = candidate
        print(f"Loaded {candidate} ok")
        break
    except Exception as exc:
        errors.append((candidate, repr(exc)))
        print(f"Load failed for {candidate}: {exc}")

if model is None:
    raise RuntimeError(
        "Both Qwen3.5 4B and Qwen3 4B failed to load via Unsloth. "
        "Investigate Unsloth version mismatch before Friday. Errors: " + str(errors)
    )

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 4. Combined teacher seed dataset (78 usable samples, 10 episodes, 2 teachers)

Loads `seed_combined.jsonl` and flattens per-step `(prompt, response_text)` pairs into chat-formatted training text.

| Teacher | Episodes | Role |
|---|---|---|
| Claude Opus 4.7 | 6 | Expert demos — author-optimal paths, mean score 0.769, all resolved |
| Llama-3.3-70B (via Fireworks) | 4 | Realistic agent — noisier, mean score 0.725, 3/4 resolved |

After filtering steps with empty prompts (a logging bug in 4 samples), we get 78 usable (prompt, action) pairs. Enough for format-anchoring SFT — the goal here is to teach the model the action-JSON shape, not to imprint a specific strategy. GRPO handles strategy.

Upload `train/data/seed_combined.jsonl` to `/content/seed_combined.jsonl` before running this cell.

In [ ]:
import json
from datasets import Dataset

SYSTEM = (
    "You are an SRE agent operating inside the sre-gym environment. "
    "On each turn, emit exactly one UnifiedIncidentAction JSON object — no surrounding prose, "
    "no code fences, no commentary. Valid action_types: query_logs, query_metrics, "
    "query_dependencies, query_deploys, rollback_deploy, restart_service, isolate_service, "
    "run_check, submit_hypothesis, escalate, declare_resolved."
)


def _load_seed_samples(path: str):
    """Flatten canonical trajectory JSONL into (prompt, response_text) pairs.

    Skips steps whose prompt is empty (a handful of rollback steps in the seed
    set lost their prior observation to a chained-call logging bug — 35 of 39
    samples are clean, which is enough for format-anchoring SFT).
    """
    samples = []
    held_out_prompt = None
    skipped = 0
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            ep = json.loads(line)
            for step in ep["trajectory"]:
                prompt = step.get("prompt") or ""
                response = step.get("response_text") or ""
                if len(prompt) < 50 or not response:
                    skipped += 1
                    continue
                samples.append((prompt, response))
            if held_out_prompt is None and ep["trajectory"]:
                first = ep["trajectory"][0]
                if first.get("prompt") and len(first["prompt"]) >= 50:
                    held_out_prompt = first["prompt"]
    print(f"loaded {len(samples)} training samples, skipped {skipped} incomplete")
    return samples, held_out_prompt


samples, HELD_OUT_PROMPT = _load_seed_samples(SEED_JSONL_PATH)

if not samples:
    raise RuntimeError(
        f"No samples found in {SEED_JSONL_PATH}. Did you upload claude_seed.jsonl to /content/?"
    )


def _format(example):
    prompt, action = example
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": action},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}


raw = [_format(ex) for ex in samples]
dataset = Dataset.from_list(raw)
print(f"dataset: {len(dataset)} rows")
print("sample text (first 600 chars):")
print(dataset[0]["text"][:600])

## 5. SFT training — 500 steps

~13 epochs over the 39 samples at `batch_size=2, grad_accum=4`. Enough for format compliance + basic strategy imprinting, not enough to overfit. On A100 40GB this is ~20 minutes wall-clock.

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=10,
    max_steps=NUM_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=42,
    report_to="wandb",
    run_name=WANDB_RUN_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
)

os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=cfg,
)

trainer_stats = trainer.train()
print(trainer_stats)

## 6. Save LoRA adapter + sanity-check inference

In [ ]:
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)

from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)

# Use the held-out prompt captured during dataset load (first step of first episode).
# Should emit a valid UnifiedIncidentAction JSON object (typically query_deploys on the
# offending service, matching what Claude did).
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": HELD_OUT_PROMPT},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=96, temperature=0.0, do_sample=False)
generated = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
print("GENERATED ACTION:")
print(generated)

import json as _json
try:
    parsed = _json.loads(generated.strip())
    assert "action_type" in parsed
    print(f"\nFORMAT OK: action_type={parsed['action_type']}")
except Exception as exc:
    print(f"\nFORMAT WARNING: {exc}. Check that SFT learned the JSON shape.")

## Verification checklist

- [ ] Cell 3 loaded a model without OOM or import errors
- [ ] Cell 4 produced 39 chat-formatted rows (no tokenizer errors, no missing JSONL file)
- [ ] Cell 5 ran 500 steps, wandb logged a decreasing loss curve (~2.0 → < 0.5)
- [ ] Cell 6 generated format-valid JSON with `action_type` field
- [ ] `/content/sanity_ckpt/adapter_model.safetensors` exists

If any box is unchecked, debug before moving to GRPO — the GRPO run will burn 4–6 hours of A100 time and any upstream format bug amplifies there.

## Next step after this notebook passes

1. Upload the LoRA adapter to `dakshdoesdev/sre-gym-qwen35-4b-sft` (use `huggingface_hub.upload_folder` or `hf upload`).
2. Clone `Gen-Verse/OpenClaw-RL` in a separate venv.
3. Apply the one-line import patch documented in `openclaw_integration/README.md`.
4. Boot `openclaw_integration/pool_server.py` on port 8100.
5. Launch OpenClaw-RL's GRPO script with `ENV_SERVER_URL=http://127.0.0.1:8100` and starting weights = this SFT checkpoint.
6. Target 500 GRPO steps.